# Imports

In [ ]:
from petrobras_dataset import read_all_wells_with_dept_to_list, filter_commom_features
from utils.training_utilities import add_derived_features, WarmupScheduler, evaluate_model
from utils.modelTrainer import load_best_configuration, FinalModelTrainer, train_model_with_validation_split
from utils.WellLogDataset import WellLogAugmentation, WellLogDataset
from datetime import datetime
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader
from selfAttention import Rebuilt_SAIDNN
import torch.nn as nn
import torch
import os
import json
import joblib
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, StepLR

# Cross-Fold Hyperparameter Experiment

In [ ]:
class CrossFoldHyperparameterExperiment:
    def __init__(self, base_config):
        self.base_config = base_config
        self.result = {}
        self.experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.results_dir = f"experiments/experiment_{self.experiment_id}"
        os.makedirs(self.results_dir, exist_ok=True)

    def define_hyperparameter_configurations(self):
        """Define LARGER model configurations"""
        configurations = [
            # {
            #     "embed_dim": 256,
            #     "num_heads": 8,
            #     "num_blocks": 4,
            #     "dropout": 0.3,
            #     "learning_rate": 0.0001,
            #     "batch_size": 16,
            #     "scheduler_type": "plateau",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adamw"
            # },
            # {
            #     "embed_dim": 256,
            #     "num_heads": 16,
            #     "num_blocks": 3,
            #     "dropout": 0.25,
            #     "learning_rate": 0.0005,
            #     "batch_size": 16,
            #     "scheduler_type": "cosine",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adamw"
            # },
            # # DEEP MODELS - More Layers
            # {
            #     "embed_dim": 128,
            #     "num_heads": 8,
            #     "num_blocks": 6,  # More blocks
            #     "dropout": 0.3,
            #     "learning_rate": 0.0001,
            #     "batch_size": 32,
            #     "scheduler_type": "plateau",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adamw"
            # },
            # {
            #     "embed_dim": 128,
            #     "num_heads": 4,
            #     "num_blocks": 5,  # More blocks
            #     "dropout": 0.25,
            #     "learning_rate": 0.0005,
            #     "batch_size": 32,
            #     "scheduler_type": "cosine",
            #     "criterion_type": "mse",
            #     "optimizer_type": "adam"
            # },
            # # MEDIUM-LARGE MODELS
            # {
            #     "embed_dim": 128,
            #     "num_heads": 8,
            #     "num_blocks": 3,
            #     "dropout": 0.2,
            #     "learning_rate": 0.0001,
            #     "batch_size": 32,
            #     "scheduler_type": "plateau",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adamw"
            # },
            # {
            #     "embed_dim": 128,
            #     "num_heads": 4,
            #     "num_blocks": 4,
            #     "dropout": 0.3,
            #     "learning_rate": 0.0005,
            #     "batch_size": 32,
            #     "scheduler_type": "cosine",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adamw"
            # },
            # {
            #     "embed_dim": 192,
            #     "num_heads": 8,
            #     "num_blocks": 3,
            #     "dropout": 0.25,
            #     "learning_rate": 0.0005,
            #     "batch_size": 24,
            #     "scheduler_type": "plateau",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adamw"
            # },
            # # BASELINE (for comparison)
            # {
            #     "embed_dim": 64,
            #     "num_heads": 4,
            #     "num_blocks": 2,
            #     "dropout": 0.2,
            #     "learning_rate": 0.001,
            #     "batch_size": 32,
            #     "scheduler_type": "plateau",
            #     "criterion_type": "huber",
            #     "optimizer_type": "adam"
            # },
            # WIDER MODELS
            {
                "embed_dim": 160,
                "num_heads": 10,  # More heads
                "num_blocks": 3,
                "dropout": 0.2,
                "learning_rate": 0.0005,
                "batch_size": 32,
                "scheduler_type": "plateau",
                "criterion_type": "huber",
                "optimizer_type": "adamw"
            },
            # {
            #     "embed_dim": 144,
            #     "num_heads": 12,  # More heads
            #     "num_blocks": 3,
            #     "dropout": 0.25,
            #     "learning_rate": 0.0003,
            #     "batch_size": 32,
            #     "scheduler_type": "cosine",
            #     "criterion_type": "mse",
            #     "optimizer_type": "adam"
            # }
        ]

        return configurations
    
    def run_cross_fold_experiments(self, feature_combinations, wells_data, target_feature="VS"):
        """Run cross-fold validation"""
        hyperparameter_configs = self.define_hyperparameter_configurations()

        wells_with_target = [df for df in wells_data if target_feature in df.columns]

        print(f"Total wells with {target_feature}: {len(wells_with_target)}")
        print(f"Running {len(hyperparameter_configs)} hyperparameter configurations")
        print(f"Testing {len(feature_combinations)} feature combinations")
        print(f"Cross-fold validation with {len(wells_with_target)} wells")
        print(
            f"Total experiments: {len(hyperparameter_configs) * len(feature_combinations) * len(wells_with_target)}"
        )

        experiment_count = 0
        total_experiments = (
            len(hyperparameter_configs)
            * len(feature_combinations)
            * len(wells_with_target)
        )

        for features_to_use in feature_combinations:
            print(f"\n{'=' * 80}")
            print(f"FEATURE COMBINATION: {features_to_use}")
            print(f"{'=' * 80}")

            for test_well_idx in range(len(wells_with_target)):
                print(f"\n{'-' * 80}")
                print(
                    f"FOLD {test_well_idx + 1}/{len(wells_with_target)}: Using Well {test_well_idx} as test set"
                )
                print(f"{'-' * 80}")

                # Split data
                train_wells_dfs = [
                    wells_with_target[i]
                    for i in range(len(wells_with_target))
                    if i != test_well_idx
                ]
                test_well_df = wells_with_target[test_well_idx]

                # Prepare training data
                combined_train_df = pd.concat(train_wells_dfs, ignore_index=True)
                combined_train_df_cleaned = combined_train_df.dropna(
                    subset=[target_feature]
                )

                # Fit scaler
                scaler = RobustScaler()
                features_for_scaling = [f for f in features_to_use if f != "DEPT"]

                if len(combined_train_df_cleaned[features_for_scaling]) == 0:
                    print(
                        f"  No valid training data for features {features_to_use}, fold {test_well_idx}"
                    )
                    continue

                scaler.fit(combined_train_df_cleaned[features_for_scaling])

                # Process training wells
                processed_train_dfs = []
                for df in train_wells_dfs:
                    processed_df = self.preprocess_df(
                        df, scaler, features_for_scaling, target_feature
                    )
                    processed_train_dfs.append(processed_df)

                all_train_features = []
                all_train_targets = []
                for df in processed_train_dfs:
                    all_train_features.extend(df[features_to_use].values.tolist())
                    all_train_targets.extend(df[target_feature].values.tolist())

                # Process test well
                processed_test_df = self.preprocess_df(
                    test_well_df, scaler, features_for_scaling, target_feature
                )
                test_features_data = processed_test_df[features_to_use].values
                test_target_data = processed_test_df[target_feature].values

                if len(all_train_features) == 0:
                    print(f"  No valid training samples for fold {test_well_idx}")
                    continue

                # Run hyperparameter configurations
                for hyperparam_config in hyperparameter_configs:
                    experiment_count += 1
                    print(f"\nExperiment {experiment_count}/{total_experiments}")
                    print(f"  Hyperparameters: {hyperparam_config}")

                    try:
                        result = self.run_single_experiment(
                            features_to_use,
                            all_train_features,
                            all_train_targets,
                            test_features_data,
                            test_target_data,
                            scaler,
                            hyperparam_config,
                            fold_idx=test_well_idx,
                        )

                        experiment_key = (
                            tuple(features_to_use),
                            tuple(hyperparam_config.items()),
                            test_well_idx,
                        )
                        self.results[experiment_key] = result

                        # Print results
                        print(f"  Results:")
                        print(f"    Test Loss: {result['test_metrics']['loss']:.6f}")
                        print(f"    Test R²: {result['test_metrics']['r2']:.4f}")
                        print(f"    Test RMSE: {result['test_metrics']['rmse']:.4f}")
                        print(f"    Test MSE: {result['test_metrics']['mse']:.4f}")
                        print(f"    Test MAE: {result['test_metrics']['mae']:.4f}")

                        # Save intermediate results
                        if experiment_count % 5 == 0:
                            self.save_results()

                    except Exception as e:
                        print(f"  Experiment failed: {e}")
                        import traceback

                        traceback.print_exc()
                        continue

        self.save_results()
        self.analyze_results()
        self.save_best_models()
        return self.results

    def run_single_experiment(
        self,
        features_to_use,
        all_train_features,
        all_train_targets,
        test_features_data,
        test_target_data,
        scaler,
        hyperparams,
        fold_idx,
    ):
        sequence_length = self.base_config["sequence_length"]
        mask_value = self.base_config["mask_value"]

        # Create augmentation
        augmentation = WellLogAugmentation(noise_level=0.01, scale_range=(0.95, 1.05))

        # Create datasets
        train_dataset = WellLogDataset(
            all_train_features,
            all_train_targets,
            sequence_length,
            mask_value,
            augmentation=augmentation,
        )
        test_dataset = WellLogDataset(
            test_features_data,
            test_target_data,
            sequence_length,
            mask_value,
            augmentation=None,
        )

        train_loader = DataLoader(
            train_dataset, batch_size=hyperparams["batch_size"], shuffle=True
        )
        test_loader = DataLoader(
            test_dataset, batch_size=hyperparams["batch_size"], shuffle=False
        )

        # Create model (using SAIDNN_v3)
        model = Rebuilt_SAIDNN(
            n_features=len(features_to_use),
            sequence_length=sequence_length,
            embed_dim=hyperparams["embed_dim"],
            num_heads=hyperparams["num_heads"],
            num_blocks=hyperparams["num_blocks"],
            dropout=hyperparams["dropout"],
            use_attention_pooling=True,
        )

        # Create criterion
        if hyperparams["criterion_type"] == "huber":
            criterion = nn.HuberLoss(delta=1.0)
        elif hyperparams["criterion_type"] == "mse":
            criterion = nn.MSELoss()
        else:  # mae
            criterion = nn.L1Loss()

        # Create optimizer
        if hyperparams["optimizer_type"] == "adam":
            optimizer = torch.optim.Adam(
                model.parameters(), lr=hyperparams["learning_rate"]
            )
        elif hyperparams["optimizer_type"] == "adamw":
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=hyperparams["learning_rate"]
            )
        else:  # sgd
            optimizer = torch.optim.SGD(
                model.parameters(), lr=hyperparams["learning_rate"], momentum=0.9
            )

        # Create scheduler
        if hyperparams["scheduler_type"] == "plateau":
            base_scheduler = ReduceLROnPlateau(
                optimizer, mode="min", factor=0.5, patience=10
            )
        elif hyperparams["scheduler_type"] == "cosine":
            base_scheduler = CosineAnnealingLR(
                optimizer, T_max=self.base_config["num_epochs"]
            )
        else:  # step
            base_scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

        # Wrap with warmup
        scheduler = WarmupScheduler(
            optimizer, warmup_epochs=5, base_scheduler=base_scheduler
        )

        # Train model
        trained_model, history, best_val_loss = train_model_with_validation_split(
            model,
            train_loader,
            criterion,
            optimizer,
            scheduler,
            self.base_config["num_epochs"],
            self.base_config["patience"],
            verbose=False,
        )

        # Evaluate
        test_metrics = evaluate_model(trained_model, test_loader, criterion)

        # Create unique identifier
        model_id = (
            f"fold{fold_idx}_features{'_'.join([f[:3] for f in features_to_use])}_"
            f"emb{hyperparams['embed_dim']}_heads{hyperparams['num_heads']}_"
            f"blocks{hyperparams['num_blocks']}"
        )

        model_path = os.path.join(self.results_dir, f"{model_id}_model.pth")
        scaler_path = os.path.join(self.results_dir, f"{model_id}_scaler.pkl")

        torch.save(trained_model.state_dict(), model_path)
        joblib.dump(scaler, scaler_path)

        result = {
            "fold_idx": fold_idx,
            "features": features_to_use,
            "hyperparams": hyperparams,
            "best_val_loss": best_val_loss,
            "test_metrics": test_metrics,
            "model_path": model_path,
            "scaler_path": scaler_path,
            "history": history,
            "final_train_loss": history["train_loss"][-1],
            "final_train_r2": history["train_r2"][-1],
            "model_params": sum(p.numel() for p in trained_model.parameters()),
            "model_id": model_id,
        }

        return result

    def preprocess_df(
        self, df, scaler, features_for_scaling, target_feature, mask_value=-1.0
    ):
        processed_df = df.copy()
        if not processed_df[features_for_scaling].empty:
            processed_df[features_for_scaling] = scaler.transform(
                processed_df[features_for_scaling]
            )
        for col in features_for_scaling:
            processed_df[col] = processed_df[col].fillna(mask_value)
        if target_feature in processed_df.columns:
            processed_df[target_feature] = processed_df[target_feature].fillna(
                mask_value
            )
        else:
            processed_df[target_feature] = mask_value
        return processed_df

    def save_results(self):
        results_path = os.path.join(self.results_dir, "experiment_results.json")

        serializable_results = {}
        for key, value in self.results.items():
            str_key = f"{key[0]}_{hash(key[1])}_fold_{key[2]}"
            serializable_value = {
                k: v for k, v in value.items() if k not in ["history"]
            }
            serializable_value["experiment_id"] = self.experiment_id
            serializable_results[str_key] = serializable_value

        with open(results_path, "w") as f:
            json.dump(serializable_results, f, indent=2, default=str)

        print(f"\nResults saved to {results_path}")

    def analyze_results(self):
        if not self.results:
            print("No results to analyze")
            return

        print(f"\n{'=' * 80}")
        print("EXPERIMENT ANALYSIS")
        print(f"{'=' * 80}")

        # Aggregate results across folds
        config_results = {}
        for key, result in self.results.items():
            features = key[0]
            hyperparams = key[1]
            config_key = (features, hyperparams)

            if config_key not in config_results:
                config_results[config_key] = {
                    "test_r2": [],
                    "test_rmse": [],
                    "test_mse": [],
                    "test_mae": [],
                    "test_loss": [],
                }

            config_results[config_key]["test_r2"].append(result["test_metrics"]["r2"])
            config_results[config_key]["test_rmse"].append(
                result["test_metrics"]["rmse"]
            )
            config_results[config_key]["test_mse"].append(result["test_metrics"]["mse"])
            config_results[config_key]["test_mae"].append(result["test_metrics"]["mae"])
            config_results[config_key]["test_loss"].append(
                result["test_metrics"]["loss"]
            )

        # Calculate average metrics
        avg_results = []
        for config_key, metrics in config_results.items():
            avg_result = {
                "features": list(config_key[0]),
                "hyperparams": dict(config_key[1]),
                "avg_test_r2": np.mean(metrics["test_r2"]),
                "std_test_r2": np.std(metrics["test_r2"]),
                "avg_test_rmse": np.mean(metrics["test_rmse"]),
                "std_test_rmse": np.std(metrics["test_rmse"]),
                "avg_test_mse": np.mean(metrics["test_mse"]),
                "std_test_mse": np.std(metrics["test_mse"]),
                "avg_test_mae": np.mean(metrics["test_mae"]),
                "std_test_mae": np.std(metrics["test_mae"]),
            }
            avg_results.append(avg_result)

        # Sort by R²
        avg_results.sort(key=lambda x: x["avg_test_r2"], reverse=True)

        print(f"\n{'=' * 80}")
        print("TOP 10 CONFIGURATIONS BY AVERAGE TEST R²")
        print(f"{'=' * 80}")

        for i, result in enumerate(avg_results[:10]):
            print(f"\n{i + 1}. Configuration:")
            print(f"   Features: {result['features']}")
            print(f"   Hyperparameters:")
            for key, value in result["hyperparams"].items():
                print(f"     {key}: {value}")
            print(
                f"   Average Test R²: {result['avg_test_r2']:.4f} ± {result['std_test_r2']:.4f}"
            )
            print(
                f"   Average Test RMSE: {result['avg_test_rmse']:.4f} ± {result['std_test_rmse']:.4f}"
            )
            print(
                f"   Average Test MSE: {result['avg_test_mse']:.4f} ± {result['std_test_mse']:.4f}"
            )
            print(
                f"   Average Test MAE: {result['avg_test_mae']:.4f} ± {result['std_test_mae']:.4f}"
            )

        # Save averaged results
        avg_results_path = os.path.join(self.results_dir, "averaged_results.json")
        with open(avg_results_path, "w") as f:
            json.dump(avg_results, f, indent=2, default=str)

        print(f"\nAveraged results saved to {avg_results_path}")

    def save_best_models(self):
        """Save information about best models from each fold"""
        if not self.results:
            print("No results to save")
            return

        # Find best model for each fold
        folds = {}
        for key, result in self.results.items():
            fold_idx = key[2]
            if fold_idx not in folds:
                folds[fold_idx] = []
            folds[fold_idx].append((key, result))

        best_models_info = {}
        for fold_idx, fold_results in folds.items():
            # Sort by test R²
            fold_results.sort(key=lambda x: x[1]["test_metrics"]["r2"], reverse=True)

            best_key, best_result = fold_results[0]

            best_models_info[f"fold_{fold_idx}"] = {
                "model_id": best_result["model_id"],
                "features": best_result["features"],
                "hyperparams": best_result["hyperparams"],
                "test_metrics": best_result["test_metrics"],
                "model_path": best_result["model_path"],
                "scaler_path": best_result["scaler_path"],
            }

        best_models_path = os.path.join(self.results_dir, "best_models_per_fold.json")
        with open(best_models_path, "w") as f:
            json.dump(best_models_info, f, indent=2, default=str)

        print(f"\n{'=' * 80}")
        print("BEST MODEL PER FOLD")
        print(f"{'=' * 80}")

        for fold_key, model_info in best_models_info.items():
            print(f"\n{fold_key.upper()}:")
            print(f"  Model ID: {model_info['model_id']}")
            print(f"  Features: {model_info['features']}")
            print(f"  Test R²: {model_info['test_metrics']['r2']:.4f}")
            print(f"  Test RMSE: {model_info['test_metrics']['rmse']:.4f}")
            print(f"  Model Path: {model_info['model_path']}")

        print(f"\nBest models info saved to {best_models_path}")


In [ ]:
import polars as pl

def _normalize_hyperparams(hyperparams):
    normalized = dict(hyperparams)
    if "embed_dim" not in normalized and "embed_dim" in normalized:
        normalized["embed_dim"] = normalized["embed_dim"]
    if "learning_rate" not in normalized and "learning_rate" in normalized:
        normalized["learning_rate"] = normalized["learning_rate"]
    return normalized

def _run_cross_fold_experiments_polars(self, feature_combinations, wells_data, target_feature="VS"):
    """Run cross-fold validation using Polars dataframes."""
    hyperparameter_configs = self.define_hyperparameter_configurations()

    if not hasattr(self, "results"):
        self.results = {}

    wells_with_target = [df for df in wells_data if target_feature in df.columns]

    print(f"Total wells with {target_feature}: {len(wells_with_target)}")
    print(f"Running {len(hyperparameter_configs)} hyperparameter configurations")
    print(f"Testing {len(feature_combinations)} feature combinations")
    print(f"Cross-fold validation with {len(wells_with_target)} wells")
    print(
        f"Total experiments: {len(hyperparameter_configs) * len(feature_combinations) * len(wells_with_target)}"
    )

    experiment_count = 0
    total_experiments = (
        len(hyperparameter_configs)
        * len(feature_combinations)
        * len(wells_with_target)
    )

    for features_to_use in feature_combinations:
        print(f"\n{'=' * 80}")
        print(f"FEATURE COMBINATION: {features_to_use}")
        print(f"{'=' * 80}")

        for test_well_idx in range(len(wells_with_target)):
            print(f"\n{'-' * 80}")
            print(
                f"FOLD {test_well_idx + 1}/{len(wells_with_target)}: Using Well {test_well_idx} as test set"
            )
            print(f"{'-' * 80}")

            train_wells_dfs = [
                wells_with_target[i]
                for i in range(len(wells_with_target))
                if i != test_well_idx
            ]
            test_well_df = wells_with_target[test_well_idx]

            combined_train_df = pl.concat(train_wells_dfs, how="vertical")
            combined_train_df_cleaned = combined_train_df.drop_nulls(subset=[target_feature])

            scaler = RobustScaler()
            features_for_scaling = [f for f in features_to_use if f != "DEPT"]

            if combined_train_df_cleaned.height == 0:
                print(
                    f"  No valid training data for features {features_to_use}, fold {test_well_idx}"
                )
                continue

            scaler.fit(combined_train_df_cleaned.select(features_for_scaling).to_numpy())

            processed_train_dfs = []
            for df in train_wells_dfs:
                processed_df = self.preprocess_df(
                    df, scaler, features_for_scaling, target_feature
                )
                processed_train_dfs.append(processed_df)

            all_train_features = []
            all_train_targets = []
            for df in processed_train_dfs:
                all_train_features.extend(df.select(features_to_use).to_numpy().tolist())
                all_train_targets.extend(df.get_column(target_feature).to_list())

            processed_test_df = self.preprocess_df(
                test_well_df, scaler, features_for_scaling, target_feature
            )
            test_features_data = processed_test_df.select(features_to_use).to_numpy()
            test_target_data = processed_test_df.get_column(target_feature).to_numpy()

            if len(all_train_features) == 0:
                print(f"  No valid training samples for fold {test_well_idx}")
                continue

            for hyperparam_config in hyperparameter_configs:
                experiment_count += 1
                normalized_hyperparam_config = _normalize_hyperparams(hyperparam_config)
                print(f"\nExperiment {experiment_count}/{total_experiments}")
                print(f"  Hyperparameters: {normalized_hyperparam_config}")

                try:
                    result = self.run_single_experiment(
                        features_to_use,
                        all_train_features,
                        all_train_targets,
                        test_features_data,
                        test_target_data,
                        scaler,
                        normalized_hyperparam_config,
                        fold_idx=test_well_idx,
                    )

                    experiment_key = (
                        tuple(features_to_use),
                        tuple(normalized_hyperparam_config.items()),
                        test_well_idx,
                    )
                    self.results[experiment_key] = result

                    print(f"  Results:")
                    print(f"    Test Loss: {result['test_metrics']['loss']:.6f}")
                    print(f"    Test R²: {result['test_metrics']['r2']:.4f}")
                    print(f"    Test RMSE: {result['test_metrics']['rmse']:.4f}")
                    print(f"    Test MSE: {result['test_metrics']['mse']:.4f}")
                    print(f"    Test MAE: {result['test_metrics']['mae']:.4f}")

                    if experiment_count % 5 == 0:
                        self.save_results()

                except Exception as e:
                    print(f"  Experiment failed: {e}")
                    import traceback

                    traceback.print_exc()
                    continue

    self.save_results()
    self.analyze_results()
    self.save_best_models()
    return self.results


def _preprocess_df_polars(
    self, df, scaler, features_for_scaling, target_feature, mask_value=-1.0
):
    processed_df = df.clone()

    if features_for_scaling and processed_df.height > 0:
        scaled_values = scaler.transform(processed_df.select(features_for_scaling).to_numpy())
        processed_df = processed_df.with_columns(
            [
                pl.Series(name=col, values=scaled_values[:, idx])
                for idx, col in enumerate(features_for_scaling)
            ]
        )

    for col in features_for_scaling:
        if col in processed_df.columns:
            processed_df = processed_df.with_columns(
                pl.col(col).fill_null(mask_value).alias(col)
            )

    if target_feature in processed_df.columns:
        processed_df = processed_df.with_columns(
            pl.col(target_feature).fill_null(mask_value).alias(target_feature)
        )
    else:
        processed_df = processed_df.with_columns(
            pl.lit(mask_value).alias(target_feature)
        )

    return processed_df


CrossFoldHyperparameterExperiment.run_cross_fold_experiments = _run_cross_fold_experiments_polars
CrossFoldHyperparameterExperiment.preprocess_df = _preprocess_df_polars

# Main pipeline execution
Improved Well Log VS Prediction Pipeline
## Configuration - Improved

In [ ]:
base_config = {
    "sequence_length": 15,
    "mask_value": -1.0,
    "num_epochs": 150,
    "patience": 20,
    "target_feature": "VS",
}

## Features Combinations - MORE FEATURES

In [ ]:
print("=" * 80)
print("IMPROVED WELL LOG VS PREDICTION PIPELINE")
print("=" * 80)
    # Feature combinations - MORE FEATURES
feature_combinations = [
        # More features = better predictions
        ["VP", "RHO", "GR", "CALIPER", "POROSIDADE", "SATURACAO", "ARGILOSIDADE"],
        ["VP", "RHO", "GR", "CALIPER", "POROSIDADE", "SATURACAO"],
        # With derived features (if available)
        ["VP", "RHO", "ACOUSTIC_IMP", "GR", "POROSIDADE", "SATURACAO"],
        # Core physics-based
        ["VP", "RHO", "POROSIDADE", "GR", "ARGILOSIDADE"],
        ["VP", "RHO", "POROSIDADE", "SATURACAO"],
    ]

## Step 1: Load Data

In [ ]:
print("\n" + "=" * 80)
print("STEP 1: LOADING WELL DATA")
print("=" * 80)

wells = read_all_wells_with_dept_to_list(features="all")
well_dfs = filter_commom_features(wells, ignore=["VS"])

### Add derived features

In [ ]:
print("Adding derived features...")
well_dfs = [add_derived_features(df) for df in well_dfs]

wells_with_vs = [df for df in well_dfs if "VS" in df.columns]
wells_without_vs = [df for df in well_dfs if "VS" not in df.columns]

print(f"Total wells loaded: {len(well_dfs)}")
print(f"Wells with VS: {len(wells_with_vs)}")
print(f"Wells without VS: {len(wells_without_vs)}")

## Step 2: Run Cross-Fold Validation

In [ ]:
print("\n" + "=" * 80)
print("STEP 2: CROSS-FOLD VALIDATION")
print("=" * 80)

existing_experiments = [d for d in os.listdir("experiments") if d.startswith("experiment_")]

if existing_experiments:
    print(f"\nFound {len(existing_experiments)} existing experiment(s):")
    for i, exp_dir in enumerate(existing_experiments):
        print(f" {i + 1}. {exp_dir}")

    response = (
        input("\nDo you want to use existing experiments(s) (y/n): ").strip().lower()
    )

    if response == "y":
        if len(existing_experiments) == 1:
            experiment_dir = existing_experiments[0]
        else:
            exp_idx = (
                int(input(f"Which experiment? (1-{len(existing_experiments)}): ").strip()) - 1
            )
            experiment_dir = existing_experiments[exp_idx]
        
        print(f"\nUsing existing experiment: {experiment_dir}")
        best_config = load_best_configuration(experiment_dir)
    else:
        print("\nRunning new cross-fold validation...")
        experiment = CrossFoldHyperparameterExperiment(base_config)
        results = experiment.run_cross_fold_experiments(feature_combinations, wells_with_vs)
        experiment_dir = experiment.results_dir
        best_config = load_best_configuration(experiment_dir)
else:
    print("\nNo existing experiments found. Running cross-fold validation...")
    experiment = CrossFoldHyperparameterExperiment(base_config)
    results = experiment.run_cross_fold_experiments(feature_combinations, wells_with_vs)
    experiment_dir = experiment.results_dir
    best_config = load_best_configuration(experiment_dir)

## STEP 3: Train Final Model

In [ ]:
print("\n" + "=" * 80)
print("STEP 3: TRAINING FINAL MODEL")
print("=" * 80)

final_output_dir = os.path.join(experiment_dir, "final_model")
trainer = FinalModelTrainer(best_config, base_config, output_dir=final_output_dir)
final_model, final_scaler, features_to_use = trainer.train_final_model(
    wells_with_vs, target_feature=base_config["target_feature"]
)

## STEP 4: Make Predictions

In [ ]:
print(f"\n" + "=" * 80)
print("STEP 4: MAKING PREDICTIONS")
print("=" * 80)

all_results = trainer.predict_on_wells(
    final_model,
    final_scaler,
    features_to_use,
    wells_without_vs,
    wells_with_vs,
    target_feature=base_config["target_feature"]
)

## STEP 5: Generate Plots and Reports

In [ ]:
print("\n" + "=" * 80)
print("STEP 5: GENERATING REPORTS AND PLOTS")
print("=" * 80)

trainer.plot_predictions(all_results)
trainer.generate_summary_report(all_results, best_config)


# Done

In [ ]:
print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"\nAll results saved to: {trainer.output_dir}/")
print(f"  - Final model: final_model.pth")
print(f"  - Scaler: final_scaler.pkl")
print(f"  - Predictions: all_predictions.json")
print(f"  - Summary: SUMMARY_REPORT.txt")
print(f"  - Plots: plots/")
print("\n" + "=" * 80)